# WEEE Hyperspectral Segmentation — SST-UNet (Patent-Aligned Pipeline)
### Spectral-Spatial Transformer UNet for E-Waste Metal Classification

**Dataset**: TECNALIA WEEE Hyperspectral Dataset (Zenodo: 12565131)
**Task**: Semantic segmentation — every pixel labeled as Background, Copper, Brass, Aluminum, Steel, or White Copper.

**Why this notebook exists:** the filed patent (IDF) describes a system that trains on patches and performs
inference via a **sliding-window + 4-way test-time-augmentation (TTA) reconstruction module**. The headline
metric quoted in the patent (mIoU ≈ 0.6135, pixel accuracy ≈ 87.4%) actually came from a *different*, simpler
full-image run in an earlier notebook. This notebook rebuilds the **patch + sliding-window + TTA pipeline that
matches the patent's technical description**, targets the same or better mIoU, and fixes the White Copper
evaluation gap (the original random/scene split never put a White Copper scene in validation, so that class
could never be scored).

Loss function matches the patent clarification correspondence: **CrossEntropy + Dice + 0.3×BCE (detection head)**
— plain cross-entropy, not the experimental Focal-loss variant.

**No numbers in this notebook are hardcoded.** Every metric printed or saved is computed from an actual run.
The optional ablation section (end of notebook) trains real reduced-schedule variants instead of reporting
made-up numbers.

## 1. Install & Imports

In [ ]:
# Install any missing packages
!pip install -q einops matplotlib scikit-learn seaborn

import os, glob, random, warnings, json, time
from collections import Counter
import numpy as np
import scipy.io as sio
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from sklearn.metrics import (
    confusion_matrix, classification_report,
    jaccard_score, f1_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/WEEE_dataset
!unzip -q "/content/drive/MyDrive/dataset_tecnalia_weee_1_0_4.zip" -d /content/WEEE_dataset

In [ ]:
DATA_DIR = "/content/WEEE_dataset/dataset/data"   # folder with .mat + _gt.png

## 2. Config

In [ ]:
# ── LABELS ──────────────────────────────────────────────────────────────────
LABEL_REMAP  = {0:0, 1:1, 2:2, 3:3, 5:4, 6:5}
CLASS_NAMES  = ['Background', 'Copper', 'Brass', 'Aluminum', 'Steel', 'White Copper']
METAL_NAMES  = CLASS_NAMES[1:]
CLASS_COLORS = [
    [50,  50,  50],
    [184, 115, 51],
    [205, 177, 90],
    [169, 169, 169],
    [160, 180, 160],
    [210, 200, 190],
]

N_BANDS     = 76
NUM_CLASSES = 6
SEED        = 42

# ── PATCH TRAINING (matches patent: patch-based training) ────────────────────
PATCH_SIZE    = 64
PATCH_STRIDE  = 32
PATCH_BATCH   = 32
PATCH_EPOCHS  = 150
PATCH_LR      = 1e-4

# ── MODEL (same config as the patch run in the original notebook) ────────────
BASE_CH     = 32
N_HEADS     = 4
TRANS_DEPTH = 3

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Config loaded.')

## 3. Dataset — Load, Fix Orientation, Remap Labels

In [ ]:
all_files = os.listdir(DATA_DIR)
mat_files = [f for f in all_files if f.endswith('.mat')]
gt_files  = {f for f in all_files if f.endswith('_gt.png')}

pairs = []
for mf in mat_files:
    base = mf.replace('.mat', '')
    gt   = base + '_gt.png'
    if gt in gt_files:
        pairs.append((mf, gt))

print(f'Found {len(pairs)} HSI + GT pairs')
for p in pairs:
    print('  ', p[0])

In [ ]:
def remap_labels(mask_np):
    """Remap original labels {0,1,2,3,5,6} -> {0,1,2,3,4,5}."""
    out = np.zeros_like(mask_np)
    for orig, new in LABEL_REMAP.items():
        out[mask_np == orig] = new
    return out


def load_pair(mat_path, gt_path):
    """
    Load one HSI cube and its GT mask.
    Returns:
        cube : float32 numpy (H, W, 76) — per-band normalized
        mask : int64  numpy (H, W)      — remapped labels 0-5
    """
    raw  = sio.loadmat(mat_path)
    cube = raw['hyperfile'].astype(np.float32)

    for b in range(cube.shape[2]):
        bmin, bmax = cube[:, :, b].min(), cube[:, :, b].max()
        cube[:, :, b] = (cube[:, :, b] - bmin) / (bmax - bmin + 1e-8)

    mask = np.array(Image.open(gt_path).convert('L'), dtype=np.int64)

    H_cube, W_cube = cube.shape[:2]
    H_mask, W_mask = mask.shape

    if (H_mask, W_mask) == (H_cube, W_cube):
        pass
    elif (H_mask, W_mask) == (W_cube, H_cube):
        mask = mask.T
    else:
        mask = np.array(
            Image.fromarray(mask.astype(np.uint8)).resize(
                (W_cube, H_cube), Image.NEAREST
            ), dtype=np.int64
        )

    mask = remap_labels(mask)
    return cube, mask


# Quick test
cube, mask = load_pair(
    os.path.join(DATA_DIR, pairs[0][0]),
    os.path.join(DATA_DIR, pairs[0][1])
)
print(f'Cube shape : {cube.shape}  dtype: {cube.dtype}')
print(f'Mask shape : {mask.shape}  dtype: {mask.dtype}')
print(f'Unique labels in this mask: {np.unique(mask)}')

## 4. Scene-Aware Stratified Split — fixes the White Copper evaluation gap

The original notebook's patch-pipeline split **deliberately excluded White Copper scenes from validation**
(comment: *"White Cu: always keep in train!"*), and the base full-image pipeline used a fully random shuffle
that also happened to never place a White Copper scene in val. Either way, White Copper support in validation
was 0 — it wasn't that the model failed to detect it, it was **never evaluated on it at all**.

Fix: build the split so **one of the two White Copper scenes is held out for validation**, while keeping
scene diversity (Steel+Al, Cu+Brass+Al) in val like the original — same 13-image pool, same ~23% val ratio,
same general strategy, just with White Copper actually included this time.

In [ ]:
# Group scenes by content (from filename)
white_cu_scenes  = [p for p in pairs if p[0].startswith('White_Cu')]
steel_al_scenes  = [p for p in pairs if p[0].startswith('Fast10_Steel_Al')]
cu_brass_scenes  = [p for p in pairs if p[0].startswith('Cu_Brass_Al') and 'MIXED' not in p[0]]
mixed_scenes     = [p for p in pairs if 'MIXED' in p[0]]
other_scenes     = [p for p in pairs if p not in white_cu_scenes + steel_al_scenes + cu_brass_scenes + mixed_scenes]

assert len(white_cu_scenes) >= 1, 'No White Copper scene found in dataset — check DATA_DIR contents.'

val_names = [
    white_cu_scenes[0][0],   # <-- the fix: guarantees White Copper is scored in validation
    steel_al_scenes[0][0],   # Steel+Al scene (same choice as the original notebook)
    cu_brass_scenes[0][0],   # Cu+Brass+Al scene (same choice as the original notebook)
]

train_pairs_patch = [p for p in pairs if p[0] not in val_names]
val_pairs_patch   = [p for p in pairs if p[0] in val_names]

print(f'Train images: {len(train_pairs_patch)}')
print(f'Val images  : {len(val_pairs_patch)}')
print()
print('Train set:')
for p in train_pairs_patch: print(f'  {p[0]}')
print('Val set (includes a White Copper scene):')
for p in val_pairs_patch:   print(f'  {p[0]}')

## 5. Patch Extraction

In [ ]:
class PatchDataset(Dataset):
    """
    Extracts fixed-size patches from HSI images using a sliding window.
    Matches the patent's patch-based training description.
    """
    def __init__(self, pairs, data_dir, patch_size=64, stride=32, augment=False):
        self.patch_size = patch_size
        self.augment    = augment
        self.patches    = []

        print(f'Extracting {patch_size}x{patch_size} patches (stride={stride})...')
        total = 0
        for mat_file, gt_file in pairs:
            cube, mask = load_pair(
                os.path.join(data_dir, mat_file),
                os.path.join(data_dir, gt_file)
            )
            H, W = cube.shape[:2]
            n_patches = 0

            for y in range(0, H - patch_size + 1, stride):
                for x in range(0, W - patch_size + 1, stride):
                    cube_p = cube[y:y+patch_size, x:x+patch_size, :]
                    mask_p = mask[y:y+patch_size, x:x+patch_size]

                    # Skip patches that are >90% background (matches original patent-run threshold)
                    bg_ratio = (mask_p == 0).mean()
                    if bg_ratio > 0.90:
                        continue

                    self.patches.append((
                        torch.from_numpy(cube_p.transpose(2,0,1).copy()),
                        torch.from_numpy(mask_p.copy())
                    ))
                    n_patches += 1

            print(f'  {mat_file[:45]}: {n_patches} patches')
            total += n_patches

        print(f'Total patches: {total}')

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        cube_t, mask_t = self.patches[idx]
        cube_t = cube_t.float()

        if self.augment:
            if random.random() > 0.5:
                cube_t = torch.flip(cube_t, dims=[-1]); mask_t = torch.flip(mask_t, dims=[-1])
            if random.random() > 0.5:
                cube_t = torch.flip(cube_t, dims=[-2]); mask_t = torch.flip(mask_t, dims=[-2])
            k = random.randint(0, 3)
            if k > 0:
                cube_t = torch.rot90(cube_t, k=k, dims=[-2,-1]); mask_t = torch.rot90(mask_t, k=k, dims=[-2,-1])
            if random.random() > 0.5:
                cube_t = torch.clamp(cube_t + torch.randn_like(cube_t) * 0.02, 0.0, 1.0)

        metals = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        for c in mask_t.unique():
            metals[c.item()] = 1.0

        return cube_t, mask_t, metals


train_patch_ds = PatchDataset(train_pairs_patch, DATA_DIR, PATCH_SIZE, PATCH_STRIDE, augment=True)
print()
val_patch_ds   = PatchDataset(val_pairs_patch, DATA_DIR, PATCH_SIZE, PATCH_STRIDE, augment=False)

train_patch_loader = DataLoader(train_patch_ds, batch_size=PATCH_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_patch_loader   = DataLoader(val_patch_ds,   batch_size=PATCH_BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f'\nTrain patches: {len(train_patch_ds)} in {len(train_patch_loader)} batches')
print(f'Val patches  : {len(val_patch_ds)}  (used only for loss monitoring during training — NOT for reported metrics)')

cb, mb, met = next(iter(train_patch_loader))
print(f'Batch cube shape: {cb.shape}')
print(f'Batch mask shape: {mb.shape}')

## 6. Class Weights (computed from the actual training patches, not assumed)

In [ ]:
patch_counts = Counter()
for _, mask_t, _ in train_patch_ds:
    for c in range(NUM_CLASSES):
        patch_counts[c] += (mask_t == c).sum().item()

total_px = sum(patch_counts.values())
print('Class distribution in training patches:')
for c, name in enumerate(CLASS_NAMES):
    cnt = patch_counts[c]
    print(f'  {name:15s}: {cnt:8,} px ({100*cnt/total_px:4.1f}%)')

patch_weights = torch.zeros(NUM_CLASSES)
for c in range(NUM_CLASSES):
    if patch_counts[c] > 0:
        patch_weights[c] = total_px / (NUM_CLASSES * patch_counts[c])
patch_weights = torch.clamp(patch_weights, max=10.0).to(device)

print('\nClass weights (informational — final loss uses plain CE per patent correspondence, see Section 8):')
for c, name in enumerate(CLASS_NAMES):
    print(f'  {name:15s}: {patch_weights[c].item():.3f}')

## 7. Novel Architecture — SST-UNet
### Spectral-Spatial Transformer UNet

Unchanged from the filed patent's architecture description:
1. **Band Attention Gate (BAG)** — learns which of the 76 spectral bands matter most.
2. **Spectral-Spatial Transformer Bottleneck (SSTB)** — multi-head attention across spatial tokens at the bottleneck.
3. **Dual output heads** — segmentation map + metal-presence detection.

In [ ]:
class ConvBlock(nn.Module):
    """Standard double-conv block with GroupNorm (more stable than BatchNorm for small batches)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.GELU(),
        )
        self.res = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.res(x)


class BandAttentionGate(nn.Module):
    """Learns a per-band attention weight before spatial processing."""
    def __init__(self, n_bands):
        super().__init__()
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(n_bands, n_bands // 4),
            nn.GELU(),
            nn.Linear(n_bands // 4, n_bands),
            nn.Sigmoid()
        )

    def forward(self, x):
        weights = self.gate(x)
        weights = weights.unsqueeze(-1).unsqueeze(-1)
        return x * weights, weights.squeeze(-1).squeeze(-1)


class SpectralSpatialTransformerLayer(nn.Module):
    """Multi-head attention over spatial tokens at the bottleneck."""
    def __init__(self, dim, n_heads=4, mlp_ratio=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = nn.MultiheadAttention(dim, n_heads, dropout=0.1, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(dim * mlp_ratio, dim),
        )

    def forward(self, x):
        x2 = self.norm1(x)
        attn_out, _ = self.attn(x2, x2, x2)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SpectralSpatialTransformerBottleneck(nn.Module):
    def __init__(self, channels, n_heads=4, depth=2, max_tokens=256):
        super().__init__()
        self.max_tokens = max_tokens
        self.layers = nn.ModuleList([
            SpectralSpatialTransformerLayer(channels, n_heads)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(channels)

    def forward(self, x):
        B, C, H, W = x.shape
        tokens = rearrange(x, 'b c h w -> b (h w) c')
        for layer in self.layers:
            tokens = layer(tokens)
        tokens = self.norm(tokens)
        out = rearrange(tokens, 'b (h w) c -> b c h w', h=H, w=W)
        return out


class SSTUNet(nn.Module):
    """Spectral-Spatial Transformer UNet (SST-UNet)."""
    def __init__(self, in_ch=76, num_classes=6, base_ch=32, n_heads=4, trans_depth=2):
        super().__init__()
        c = base_ch

        self.band_gate = BandAttentionGate(in_ch)

        self.enc1 = ConvBlock(in_ch, c)
        self.enc2 = ConvBlock(c,     c*2)
        self.enc3 = ConvBlock(c*2,   c*4)
        self.pool = nn.MaxPool2d(2)

        self.bottleneck_conv = ConvBlock(c*4, c*8)
        self.transformer = SpectralSpatialTransformerBottleneck(
            channels=c*8, n_heads=n_heads, depth=trans_depth
        )

        self.up3   = nn.ConvTranspose2d(c*8, c*4, 2, stride=2)
        self.dec3  = ConvBlock(c*8, c*4)

        self.up2   = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.dec2  = ConvBlock(c*4, c*2)

        self.up1   = nn.ConvTranspose2d(c*2, c, 2, stride=2)
        self.dec1  = ConvBlock(c*2, c)

        self.seg_head = nn.Conv2d(c, num_classes, 1)

        self.det_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(c*8, 64),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
            nn.Sigmoid()
        )

    def forward(self, x):
        x, band_weights = self.band_gate(x)

        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        s3 = self.enc3(self.pool(s2))

        b  = self.bottleneck_conv(self.pool(s3))
        b  = self.transformer(b)

        det = self.det_head(b)

        d3 = self.dec3(torch.cat([self.up3(b), s3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), s2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), s1], dim=1))

        seg = self.seg_head(d1)
        return seg, det, band_weights


patch_model = SSTUNet(
    in_ch=N_BANDS, num_classes=NUM_CLASSES,
    base_ch=BASE_CH, n_heads=N_HEADS, trans_depth=TRANS_DEPTH
).to(device)

total_params = sum(p.numel() for p in patch_model.parameters() if p.requires_grad)
print(f'SST-UNet parameters: {total_params:,}')

with torch.no_grad():
    dummy = torch.randn(PATCH_BATCH, N_BANDS, PATCH_SIZE, PATCH_SIZE).to(device)
    seg_out, det_out, bw = patch_model(dummy)
    print(f'Segmentation output : {seg_out.shape}')
    print(f'Detection output    : {det_out.shape}')
    print(f'Band weights shape  : {bw.shape}')
    del dummy
    torch.cuda.empty_cache()
print('Model OK!')

## 8. Loss Function & Metrics

Per the patent clarification correspondence, the reported pipeline uses **plain CrossEntropy + Dice + 0.3×BCE**
(the Focal-loss variant explored elsewhere in the project was experimental, not the reported one). Keeping that
here so the loss formulation matches what's on file.

In [ ]:
class DiceLoss(nn.Module):
    """Dice loss — better than cross-entropy alone for imbalanced segmentation."""
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target, num_classes):
        pred_soft = F.softmax(pred, dim=1)
        target_oh = F.one_hot(target, num_classes)
        target_oh = target_oh.permute(0,3,1,2).float()
        intersection = (pred_soft * target_oh).sum(dim=(2,3))
        union = pred_soft.sum(dim=(2,3)) + target_oh.sum(dim=(2,3))
        dice = (2 * intersection + self.smooth) / (union + self.smooth)
        return 1 - dice.mean()


class CombinedLoss(nn.Module):
    """Cross-entropy + Dice + Binary cross-entropy for detection (matches patent correspondence)."""
    def __init__(self, num_classes, ce_weight=1.0, dice_weight=1.0, det_weight=0.3):
        super().__init__()
        self.ce     = nn.CrossEntropyLoss()
        self.dice   = DiceLoss()
        self.bce    = nn.BCELoss()
        self.nc     = num_classes
        self.w_ce   = ce_weight
        self.w_dice = dice_weight
        self.w_det  = det_weight

    def forward(self, seg_pred, seg_target, det_pred, det_target):
        l_ce   = self.ce(seg_pred, seg_target)
        l_dice = self.dice(seg_pred, seg_target, self.nc)
        l_det  = self.bce(det_pred, det_target)
        total  = self.w_ce * l_ce + self.w_dice * l_dice + self.w_det * l_det
        return total, l_ce, l_dice, l_det


def compute_miou(pred_mask, true_mask, num_classes):
    """Mean IoU across classes present in ground truth (absent classes excluded, not scored as 1)."""
    ious = []
    for c in range(num_classes):
        pred_c = (pred_mask == c)
        true_c = (true_mask == c)
        if true_c.sum() == 0 and pred_c.sum() == 0:
            continue
        intersection = (pred_c & true_c).sum()
        union        = (pred_c | true_c).sum()
        ious.append(intersection / (union + 1e-8))
    return np.mean(ious) if ious else 0.0


patch_criterion = CombinedLoss(NUM_CLASSES)
print('Loss function ready: CrossEntropy + Dice + 0.3xBCE')

## 9. Training — Patch-Based, Warmup + Cosine Decay

In [ ]:
opt_p = torch.optim.AdamW(patch_model.parameters(), lr=PATCH_LR, weight_decay=1e-4)

def lr_lambda(ep):
    if ep < 10: return (ep + 1) / 10
    prog = (ep - 10) / (PATCH_EPOCHS - 10)
    return 0.5 * (1 + np.cos(np.pi * prog))
sched_p = torch.optim.lr_scheduler.LambdaLR(opt_p, lr_lambda)

best_patch_miou = 0.0
best_patch_path = '/content/best_patch_sst_unet.pth'
patch_history = {'train_loss': [], 'val_patch_miou': [], 'val_patch_acc': []}

print(f'Starting patch training: {PATCH_EPOCHS} epochs, {len(train_patch_loader)} batches/epoch')
print('-' * 60)

for epoch in range(1, PATCH_EPOCHS + 1):
    patch_model.train()
    ep_loss = 0
    for cubes, masks, metals in train_patch_loader:
        cubes, masks, metals = cubes.to(device), masks.to(device), metals.to(device)
        opt_p.zero_grad()
        seg_p, det_p, _ = patch_model(cubes)
        loss, _, _, _ = patch_criterion(seg_p, masks, det_p, metals)
        loss.backward()
        nn.utils.clip_grad_norm_(patch_model.parameters(), 1.0)
        opt_p.step()
        ep_loss += loss.item()
    sched_p.step()

    # Patch-level validation loss/acc is ONLY for monitoring training — never reported as the model's mIoU
    patch_model.eval()
    val_preds, val_trues = [], []
    with torch.no_grad():
        for cubes, masks, metals in val_patch_loader:
            cubes = cubes.to(device)
            seg_p, _, _ = patch_model(cubes)
            pred = seg_p.argmax(dim=1).cpu().numpy()
            val_preds.append(pred.flatten())
            val_trues.append(masks.numpy().flatten())
    val_preds = np.concatenate(val_preds)
    val_trues = np.concatenate(val_trues)
    val_acc = (val_preds == val_trues).mean()
    ious = []
    for c in range(NUM_CLASSES):
        pc, tc = (val_preds == c), (val_trues == c)
        if tc.sum() == 0: continue
        ious.append((pc & tc).sum() / ((pc | tc).sum() + 1e-8))
    val_patch_miou = np.mean(ious)

    patch_history['train_loss'].append(ep_loss / len(train_patch_loader))
    patch_history['val_patch_miou'].append(val_patch_miou)
    patch_history['val_patch_acc'].append(val_acc)

    # Model selection uses patch-level mIoU as a fast proxy; the REPORTED metric
    # (Section 11) is always the full-image sliding-window+TTA reconstruction.
    if val_patch_miou > best_patch_miou:
        best_patch_miou = val_patch_miou
        torch.save(patch_model.state_dict(), best_patch_path)
        flag = '  <- best (patch proxy)'
    else:
        flag = ''

    if epoch % 10 == 0 or epoch == 1:
        lr_now = opt_p.param_groups[0]['lr']
        print(f'Ep {epoch:3d}/{PATCH_EPOCHS}  loss={ep_loss/len(train_patch_loader):.4f}  '
              f'patch_mIoU={val_patch_miou:.4f}  patch_acc={val_acc:.4f}  lr={lr_now:.2e}{flag}')

print(f'\nBest PATCH-level mIoU (monitoring only, not the reported metric): {best_patch_miou:.4f}')
print('See Section 11 for the actual reported full-image mIoU.')

## 10. Sliding-Window + TTA Full-Image Inference

This is the inference module described in the patent's independent claim: 64x64 patches, stride 32,
overlapping-region soft-vote averaging, 4-way flip test-time augmentation.

In [ ]:
def predict_full_image(model, cube_np, patch_size=64, stride=32, tta=True):
    """
    Sliding-window inference on a full HSI cube with optional 4-way flip TTA.
    Returns soft probability map (H, W, C) and hard prediction (H, W).
    """
    model.eval()
    H, W, B = cube_np.shape
    prob_map  = np.zeros((H, W, NUM_CLASSES), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)

    tta_transforms = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[-1]),
        lambda x: torch.flip(x, dims=[-2]),
        lambda x: torch.flip(x, dims=[-1, -2]),
    ] if tta else [lambda x: x]

    tta_inv = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[-1]),
        lambda x: torch.flip(x, dims=[-2]),
        lambda x: torch.flip(x, dims=[-1, -2]),
    ]

    patches, positions = [], []
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            patch = cube_np[y:y+patch_size, x:x+patch_size, :]
            patches.append(torch.from_numpy(patch.transpose(2,0,1).copy()).float())
            positions.append((y, x))

    for y in range(0, H - patch_size + 1, stride):
        x = W - patch_size
        if (y, x) not in positions:
            patch = cube_np[y:y+patch_size, x:x+patch_size, :]
            patches.append(torch.from_numpy(patch.transpose(2,0,1).copy()).float())
            positions.append((y, x))
    for x in range(0, W - patch_size + 1, stride):
        y = H - patch_size
        if (y, x) not in positions:
            patch = cube_np[y:y+patch_size, x:x+patch_size, :]
            patches.append(torch.from_numpy(patch.transpose(2,0,1).copy()).float())
            positions.append((y, x))

    INFER_BATCH = 64
    with torch.no_grad():
        for i in range(0, len(patches), INFER_BATCH):
            batch = torch.stack(patches[i:i+INFER_BATCH]).to(device)
            batch_positions = positions[i:i+INFER_BATCH]

            avg_prob = torch.zeros(len(batch), NUM_CLASSES, patch_size, patch_size).to(device)
            for tf, inv_tf in zip(tta_transforms, tta_inv):
                aug_batch = tf(batch)
                seg_out, _, _ = model(aug_batch)
                prob = F.softmax(seg_out, dim=1)
                avg_prob += inv_tf(prob)
            avg_prob /= len(tta_transforms)

            avg_prob_np = avg_prob.cpu().numpy()
            for j, (py, px) in enumerate(batch_positions):
                prob_map[py:py+patch_size, px:px+patch_size, :] += avg_prob_np[j].transpose(1,2,0)
                count_map[py:py+patch_size, px:px+patch_size]   += 1

    count_map = np.maximum(count_map, 1)
    prob_map /= count_map[:, :, np.newaxis]
    pred_map  = prob_map.argmax(axis=2).astype(np.int64)
    return prob_map, pred_map


print('Sliding-window + TTA inference function ready.')

## 11. Reported Evaluation — Full-Image Reconstruction (not patches)

This fixes the evaluation bug: metrics are computed on full reconstructed images via sliding-window + TTA,
not on raw validation patches. This is also what makes the White Copper fix in Section 4 actually testable —
it now has a scene in validation to be scored against.

In [ ]:
patch_model.load_state_dict(torch.load(best_patch_path, map_location=device))
patch_model.eval()

all_preds, all_trues = [], []
per_image_results = []

for mat_file, gt_file in val_pairs_patch:
    cube, mask = load_pair(os.path.join(DATA_DIR, mat_file), os.path.join(DATA_DIR, gt_file))
    _, pred = predict_full_image(patch_model, cube, PATCH_SIZE, PATCH_STRIDE, tta=True)

    img_miou = compute_miou(pred, mask, NUM_CLASSES)
    img_acc  = (pred == mask).mean()
    classes_present = sorted(np.unique(mask).tolist())

    per_image_results.append({
        'file': mat_file, 'miou': float(img_miou), 'acc': float(img_acc),
        'classes_present': [CLASS_NAMES[c] for c in classes_present]
    })
    print(f'{mat_file[:50]:52s} mIoU={img_miou:.4f}  acc={img_acc:.4f}  classes={[CLASS_NAMES[c] for c in classes_present]}')

    all_preds.append(pred.flatten())
    all_trues.append(mask.flatten())

all_preds = np.concatenate(all_preds)
all_trues = np.concatenate(all_trues)

reported_miou = compute_miou(all_preds, all_trues, NUM_CLASSES)
reported_acc  = (all_preds == all_trues).mean()

print()
print('=' * 60)
print(f'REPORTED full-image mIoU (aggregate, all val pixels): {reported_miou:.4f}')
print(f'REPORTED pixel accuracy:                              {reported_acc:.4f}')
print('=' * 60)

print()
print('=== CLASSIFICATION REPORT (per-pixel, full reconstructed images) ===')
print(classification_report(all_trues, all_preds, target_names=CLASS_NAMES, zero_division=0))

white_cu_idx = CLASS_NAMES.index('White Copper')
wc_support = (all_trues == white_cu_idx).sum()
print(f'White Copper pixels in validation set: {wc_support:,} '
      f'({"scored" if wc_support > 0 else "STILL NOT PRESENT — check split"})')

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

def compute_miou_github_style(y_true, y_pred, num_classes):
    """
    Official Tecnalia GitHub evaluation:
    - Build one global confusion matrix
    - Compute IoU for every class
    - Average over classes
    """

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=np.arange(num_classes)
    )

    ious = []

    print("="*80)
    print("Per-class IoU (GitHub style)")
    print("="*80)

    for c in range(num_classes):

        TP = cm[c, c]
        FP = cm[:, c].sum() - TP
        FN = cm[c, :].sum() - TP

        denom = TP + FP + FN

        if denom == 0:
            iou = np.nan
        else:
            iou = TP / denom

        ious.append(iou)

        name = CLASS_NAMES[c] if "CLASS_NAMES" in globals() else f"Class {c}"
        print(f"{name:15s}: {iou:.6f}")

    miou = np.nanmean(ious)

    print("="*80)
    print(f"GitHub-style mIoU : {miou:.6f}")
    print("="*80)

    return miou

In [ ]:
print("="*80)
print("Comparing mIoU implementations")
print("="*80)

my_miou = compute_miou(
    all_preds,
    all_trues,
    NUM_CLASSES
)

github_miou = compute_miou_github_style(
    all_trues,
    all_preds,
    NUM_CLASSES
)

print()
print(f"My implementation      : {my_miou:.8f}")
print(f"GitHub implementation  : {github_miou:.8f}")
print(f"Difference             : {abs(my_miou-github_miou):.12f}")

## 12. Training Curves & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(patch_history['train_loss'], color='steelblue')
axes[0].set_title('Training Loss'); axes[0].grid(alpha=0.3)
axes[1].plot(patch_history['val_patch_miou'], color='green')
axes[1].set_title('Validation Patch mIoU (training-time proxy only)'); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

cm = confusion_matrix(all_trues, all_preds, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-8)
plt.figure(figsize=(7,6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Normalized Confusion Matrix (full-image, reported)')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# QUALITATIVE RESULTS: INPUT vs GROUND TRUTH vs SST-UNET
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
from matplotlib.colors import ListedColormap

# ------------------------------------------------------------
# 1. Class definitions
# ------------------------------------------------------------
CLASS_NAMES = [
    "Background",
    "Copper",
    "Brass",
    "Aluminum",
    "Steel",
    "White Copper"
]

# Same class colours used for all masks
CLASS_COLORS = [
    "#808080",   # Background
    "#D95F02",   # Copper
    "#E6AB02",   # Brass
    "#1B9E77",   # Aluminum
    "#377EB8",   # Steel
    "#984EA3"    # White Copper
]

cmap = ListedColormap(CLASS_COLORS)

# ------------------------------------------------------------
# 2. Load the BEST SST-UNet checkpoint
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SSTUNet(
    in_channels=76,
    num_classes=6
).to(device)

checkpoint_path = "/content/best_patch_sst_unet.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

# Handle either a raw state_dict or a checkpoint dictionary
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    model.load_state_dict(checkpoint["state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

print("SST-UNet checkpoint loaded successfully.")

# ------------------------------------------------------------
# 3. Helper: create a false-RGB image from the HSI cube
# ------------------------------------------------------------
def make_false_rgb(hsi):
    """
    Create a false-RGB visualization from the hyperspectral cube.

    hsi shape:
        H x W x 76
    """

    # Same type of band selection used for visualization
    rgb = hsi[:, :, [50, 30, 10]].astype(np.float32)

    # Normalize each channel independently for visualization
    for c in range(3):
        channel = rgb[:, :, c]
        lo = np.percentile(channel, 1)
        hi = np.percentile(channel, 99)

        rgb[:, :, c] = np.clip(
            (channel - lo) / (hi - lo + 1e-8),
            0,
            1
        )

    return rgb


# ------------------------------------------------------------
# 4. Generate qualitative results for all validation scenes
# ------------------------------------------------------------

# Use the validation scenes already defined in your notebook.
# This assumes your notebook contains:
#
#     val_files
#
# and the corresponding ground-truth loading code used earlier.

for scene_idx, scene_path in enumerate(val_files):

    print(f"\nProcessing validation scene {scene_idx + 1}/{len(val_files)}")
    print(scene_path)

    # --------------------------------------------------------
    # Load HSI and ground truth
    # --------------------------------------------------------
    hsi, gt = load_scene(scene_path)

    # Make sure labels are integer class IDs
    gt = gt.astype(np.int64)

    # --------------------------------------------------------
    # Full-image prediction
    #
    # Uses your existing sliding-window inference function
    # with four-way TTA.
    # --------------------------------------------------------
    pred = predict_full_image(
        model,
        hsi,
        device=device,
        tta=True
    )

    pred = pred.astype(np.int64)

    # --------------------------------------------------------
    # False RGB visualization
    # --------------------------------------------------------
    false_rgb = make_false_rgb(hsi)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 5)
    )

    # HSI input
    axes[0].imshow(false_rgb)
    axes[0].set_title("HSI Input", fontsize=13)
    axes[0].axis("off")

    # Ground truth
    axes[1].imshow(gt, cmap=cmap, vmin=0, vmax=5)
    axes[1].set_title("Ground Truth", fontsize=13)
    axes[1].axis("off")

    # Prediction
    axes[2].imshow(pred, cmap=cmap, vmin=0, vmax=5)
    axes[2].set_title("SST-UNet Prediction", fontsize=13)
    axes[2].axis("off")

    # --------------------------------------------------------
    # Shared legend
    # --------------------------------------------------------
    handles = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            facecolor=CLASS_COLORS[i],
            label=CLASS_NAMES[i]
        )
        for i in range(len(CLASS_NAMES))
    ]

    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=3,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.5, -0.02)
    )

    fig.suptitle(
        f"Validation Scene {scene_idx + 1}",
        fontsize=15,
        y=1.02
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Save high-resolution figure
    # --------------------------------------------------------
    output_name = f"/content/SSTUNet_qualitative_scene_{scene_idx + 1}.png"

    plt.savefig(
        output_name,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(f"Saved: {output_name}")

In [ ]:
# ============================================================
# QUALITATIVE RESULTS: INPUT vs GROUND TRUTH vs SST-UNET
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import torch
from matplotlib.colors import ListedColormap

# ------------------------------------------------------------
# 1. Class definitions
# ------------------------------------------------------------
CLASS_NAMES = [
    "Background",
    "Copper",
    "Brass",
    "Aluminum",
    "Steel",
    "White Copper"
]

# Same class colours used for all masks
CLASS_COLORS = [
    "#808080",   # Background
    "#D95F02",   # Copper
    "#E6AB02",   # Brass
    "#1B9E77",   # Aluminum
    "#377EB8",   # Steel
    "#984EA3"    # White Copper
]

cmap = ListedColormap(CLASS_COLORS)

# ------------------------------------------------------------
# 2. Load the BEST SST-UNet checkpoint
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SSTUNet(
    in_channels=76,
    num_classes=6
).to(device)

checkpoint_path = "/content/best_patch_sst_unet.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

# Handle either a raw state_dict or a checkpoint dictionary
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    model.load_state_dict(checkpoint["state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

print("SST-UNet checkpoint loaded successfully.")

# ------------------------------------------------------------
# 3. Helper: create a false-RGB image from the HSI cube
# ------------------------------------------------------------
def make_false_rgb(hsi):
    """
    Create a false-RGB visualization from the hyperspectral cube.

    hsi shape:
        H x W x 76
    """

    # Same type of band selection used for visualization
    rgb = hsi[:, :, [50, 30, 10]].astype(np.float32)

    # Normalize each channel independently for visualization
    for c in range(3):
        channel = rgb[:, :, c]
        lo = np.percentile(channel, 1)
        hi = np.percentile(channel, 99)

        rgb[:, :, c] = np.clip(
            (channel - lo) / (hi - lo + 1e-8),
            0,
            1
        )

    return rgb


# ------------------------------------------------------------
# 4. Generate qualitative results for all validation scenes
# ------------------------------------------------------------

# Use the validation scenes already defined in your notebook.
# This assumes your notebook contains:
#
#     val_files
#
# and the corresponding ground-truth loading code used earlier.

for scene_idx, scene_path in enumerate(val_files):

    print(f"\nProcessing validation scene {scene_idx + 1}/{len(val_files)}")
    print(scene_path)

    # --------------------------------------------------------
    # Load HSI and ground truth
    # --------------------------------------------------------
    hsi, gt = load_scene(scene_path)

    # Make sure labels are integer class IDs
    gt = gt.astype(np.int64)

    # --------------------------------------------------------
    # Full-image prediction
    #
    # Uses your existing sliding-window inference function
    # with four-way TTA.
    # --------------------------------------------------------
    pred = predict_full_image(
        model,
        hsi,
        device=device,
        tta=True
    )

    pred = pred.astype(np.int64)

    # --------------------------------------------------------
    # False RGB visualization
    # --------------------------------------------------------
    false_rgb = make_false_rgb(hsi)

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(12, 5)
    )

    # HSI input
    axes[0].imshow(false_rgb)
    axes[0].set_title("HSI Input", fontsize=13)
    axes[0].axis("off")

    # Ground truth
    axes[1].imshow(gt, cmap=cmap, vmin=0, vmax=5)
    axes[1].set_title("Ground Truth", fontsize=13)
    axes[1].axis("off")

    # Prediction
    axes[2].imshow(pred, cmap=cmap, vmin=0, vmax=5)
    axes[2].set_title("SST-UNet Prediction", fontsize=13)
    axes[2].axis("off")

    # --------------------------------------------------------
    # Shared legend
    # --------------------------------------------------------
    handles = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            facecolor=CLASS_COLORS[i],
            label=CLASS_NAMES[i]
        )
        for i in range(len(CLASS_NAMES))
    ]

    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=3,
        frameon=False,
        fontsize=10,
        bbox_to_anchor=(0.5, -0.02)
    )

    fig.suptitle(
        f"Validation Scene {scene_idx + 1}",
        fontsize=15,
        y=1.02
    )

    plt.tight_layout()

    # --------------------------------------------------------
    # Save high-resolution figure
    # --------------------------------------------------------
    output_name = f"/content/SSTUNet_qualitative_scene_{scene_idx + 1}.png"

    plt.savefig(
        output_name,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(f"Saved: {output_name}")

## 13. Save Artifacts

In [ ]:
summary = {
    'reported_full_image_miou': float(reported_miou),
    'reported_pixel_accuracy':  float(reported_acc),
    'best_patch_level_miou_training_proxy': float(best_patch_miou),
    'per_image_results': per_image_results,
    'val_scenes': val_names,
    'train_scenes': [p[0] for p in train_pairs_patch],
    'white_copper_support_pixels': int(wc_support),
    'config': {
        'PATCH_SIZE': PATCH_SIZE, 'PATCH_STRIDE': PATCH_STRIDE, 'PATCH_EPOCHS': PATCH_EPOCHS,
        'BASE_CH': BASE_CH, 'N_HEADS': N_HEADS, 'TRANS_DEPTH': TRANS_DEPTH,
        'loss': 'CrossEntropy + Dice + 0.3xBCE', 'tta': '4-way flip', 'inference': 'sliding-window, stride=32',
    }
}

with open('/content/run_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Saved /content/best_patch_sst_unet.pth')
print('Saved /content/run_summary.json')
print()
print(json.dumps(summary, indent=2)[:1000])

In [ ]:
# ============================================================
# QUALITATIVE RESULTS: HSI vs GROUND TRUTH vs PREDICTION vs ERROR
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from matplotlib.colors import ListedColormap

# ------------------------------------------------------------
# 1. Class definitions
# ------------------------------------------------------------

CLASS_NAMES = [
    "Background",
    "Copper",
    "Brass",
    "Aluminum",
    "Steel",
    "White Copper"
]

CLASS_COLORS = [
    "#808080",   # Background
    "#D95F02",   # Copper
    "#E6AB02",   # Brass
    "#1B9E77",   # Aluminum
    "#377EB8",   # Steel
    "#984EA3"    # White Copper
]

cmap = ListedColormap(CLASS_COLORS)

# ------------------------------------------------------------
# 2. Load BEST SST-UNet checkpoint
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# IMPORTANT:
# Your actual SSTUNet constructor uses in_ch, NOT in_channels.
model = SSTUNet(
    in_ch=N_BANDS,
    num_classes=NUM_CLASSES,
    base_ch=BASE_CH,
    n_heads=N_HEADS,
    trans_depth=TRANS_DEPTH
).to(device)

checkpoint_path = best_patch_path   # already defined earlier

state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(state_dict)
model.eval()

print("SST-UNet checkpoint loaded successfully.")
print(f"Checkpoint: {checkpoint_path}")

# ------------------------------------------------------------
# 3. False-RGB function
# ------------------------------------------------------------

def make_false_rgb(hsi):

    # Use bands 50, 30, 10 exactly as in your earlier visualization
    rgb = hsi[:, :, [50, 30, 10]].astype(np.float32)

    # Normalize each channel independently
    for c in range(3):
        channel = rgb[:, :, c]

        lo = np.percentile(channel, 1)
        hi = np.percentile(channel, 99)

        rgb[:, :, c] = np.clip(
            (channel - lo) / (hi - lo + 1e-8),
            0,
            1
        )

    return rgb


# ------------------------------------------------------------
# 4. Generate qualitative results for validation scenes
# ------------------------------------------------------------

output_dir = "/content/qualitative_results"
os.makedirs(output_dir, exist_ok=True)

print("\nGenerating qualitative outputs...")
print("=" * 70)

for scene_idx, (mat_file, gt_file) in enumerate(val_pairs_patch):

    print(f"\nProcessing scene {scene_idx + 1}/{len(val_pairs_patch)}")
    print(f"HSI : {mat_file}")
    print(f"GT  : {gt_file}")

    # --------------------------------------------------------
    # Load scene
    # --------------------------------------------------------

    cube, gt = load_pair(
        os.path.join(DATA_DIR, mat_file),
        os.path.join(DATA_DIR, gt_file)
    )

    gt = gt.astype(np.int64)

    # --------------------------------------------------------
    # Full-image prediction
    # SAME protocol as reported evaluation:
    # 64x64 patches + stride 32 + 4-way TTA
    # --------------------------------------------------------

    _, pred = predict_full_image(
        model,
        cube,
        PATCH_SIZE,
        PATCH_STRIDE,
        tta=True
    )

    pred = pred.astype(np.int64)

    # --------------------------------------------------------
    # Calculate error map
    # --------------------------------------------------------

    error_map = (pred != gt).astype(np.uint8)

    # --------------------------------------------------------
    # False RGB
    # --------------------------------------------------------

    false_rgb = make_false_rgb(cube)

    # --------------------------------------------------------
    # Calculate scene metrics
    # --------------------------------------------------------

    scene_miou = compute_miou(
        pred,
        gt,
        NUM_CLASSES
    )

    scene_acc = (pred == gt).mean()

    print(f"mIoU       : {scene_miou:.4f}")
    print(f"Pixel Acc. : {scene_acc:.4f}")

    # --------------------------------------------------------
    # Plot: HSI | Ground Truth | Prediction | Error
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(16, 4.5)
    )

    # HSI false RGB
    axes[0].imshow(false_rgb)
    axes[0].set_title("False-Color HSI")
    axes[0].axis("off")

    # Ground truth
    axes[1].imshow(
        gt,
        cmap=cmap,
        vmin=0,
        vmax=NUM_CLASSES - 1
    )
    axes[1].set_title("Ground Truth")
    axes[1].axis("off")

    # Prediction
    axes[2].imshow(
        pred,
        cmap=cmap,
        vmin=0,
        vmax=NUM_CLASSES - 1
    )
    axes[2].set_title("SST-UNet Prediction")
    axes[2].axis("off")

    # Error map
    axes[3].imshow(
        error_map,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[3].set_title("Error Map")
    axes[3].axis("off")

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    handles = [
        plt.Rectangle(
            (0, 0),
            1,
            1,
            facecolor=CLASS_COLORS[i],
            label=CLASS_NAMES[i]
        )
        for i in range(NUM_CLASSES)
    ]

    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=6,
        frameon=False,
        fontsize=9,
        bbox_to_anchor=(0.5, -0.01)
    )

    # --------------------------------------------------------
    # Figure title
    # --------------------------------------------------------

    scene_name = os.path.splitext(mat_file)[0]

    fig.suptitle(
        f"{scene_name}\n"
        f"Full-image mIoU = {scene_miou:.4f}, "
        f"Pixel Accuracy = {scene_acc:.4f}",
        fontsize=12
    )

    plt.tight_layout(rect=[0, 0.08, 1, 0.92])

    # --------------------------------------------------------
    # Save high-resolution figure
    # --------------------------------------------------------

    save_path = os.path.join(
        output_dir,
        f"qualitative_scene_{scene_idx + 1}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

    print(f"Saved: {save_path}")


print("\n" + "=" * 70)
print("ALL QUALITATIVE OUTPUTS GENERATED")
print(f"Saved in: {output_dir}")
print("=" * 70)

In [ ]:
# ============================================================
# SAVE FINAL SST-UNET RESULTS FOR EXACT REPRODUCTION
# ============================================================

import os
import json
import shutil
import numpy as np
import torch

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

SAVE_DIR = "/content/SSTUNet_final_reproducible"
os.makedirs(SAVE_DIR, exist_ok=True)

# ------------------------------------------------------------
# Save exact model checkpoint
# ------------------------------------------------------------

checkpoint_save_path = os.path.join(
    SAVE_DIR,
    "SSTUNet_final_checkpoint.pth"
)

torch.save(
    model.state_dict(),
    checkpoint_save_path
)

print("Saved checkpoint:")
print(checkpoint_save_path)


# ------------------------------------------------------------
# Save configuration used for inference
# ------------------------------------------------------------

config = {
    "N_BANDS": int(N_BANDS),
    "NUM_CLASSES": int(NUM_CLASSES),
    "PATCH_SIZE": int(PATCH_SIZE),
    "PATCH_STRIDE": int(PATCH_STRIDE),
    "BASE_CH": int(BASE_CH),
    "N_HEADS": int(N_HEADS),
    "TRANS_DEPTH": int(TRANS_DEPTH),
    "TTA": True,
    "class_names": CLASS_NAMES
}

with open(
    os.path.join(SAVE_DIR, "inference_config.json"),
    "w"
) as f:
    json.dump(config, f, indent=4)


# ------------------------------------------------------------
# Save predictions and ground truth for EVERY validation scene
# ------------------------------------------------------------

all_metrics = []

model.eval()

for scene_idx, (mat_file, gt_file) in enumerate(val_pairs_patch):

    print(f"\nSaving scene {scene_idx + 1}/{len(val_pairs_patch)}")
    print(mat_file)

    # Load scene
    cube, gt = load_pair(
        os.path.join(DATA_DIR, mat_file),
        os.path.join(DATA_DIR, gt_file)
    )

    gt = gt.astype(np.int64)

    # Full-image prediction
    with torch.no_grad():

        _, pred = predict_full_image(
            model,
            cube,
            PATCH_SIZE,
            PATCH_STRIDE,
            tta=True
        )

    pred = pred.astype(np.int64)

    # Calculate metrics
    miou = compute_miou(
        pred,
        gt,
        NUM_CLASSES
    )

    accuracy = np.mean(pred == gt)

    # Save everything needed to reproduce the result
    save_file = os.path.join(
        SAVE_DIR,
        f"scene_{scene_idx + 1}.npz"
    )

    np.savez_compressed(
        save_file,
        prediction=pred,
        ground_truth=gt,
        scene_name=mat_file,
        miou=miou,
        pixel_accuracy=accuracy
    )

    all_metrics.append({
        "scene": mat_file,
        "mIoU": float(miou),
        "pixel_accuracy": float(accuracy)
    })

    print(f"mIoU       = {miou:.4f}")
    print(f"Pixel Acc. = {accuracy:.4f}")
    print(f"Saved      = {save_file}")


# ------------------------------------------------------------
# Save metrics
# ------------------------------------------------------------

aggregate_miou = np.mean(
    [x["mIoU"] for x in all_metrics]
)

aggregate_accuracy = np.mean(
    [x["pixel_accuracy"] for x in all_metrics]
)

results = {
    "scene_results": all_metrics,
    "mean_scene_mIoU": float(aggregate_miou),
    "mean_scene_pixel_accuracy": float(aggregate_accuracy)
}

with open(
    os.path.join(SAVE_DIR, "final_results.json"),
    "w"
) as f:
    json.dump(results, f, indent=4)


print("\n" + "=" * 70)
print("FINAL RESULTS SAVED")
print("=" * 70)

for r in all_metrics:
    print(
        f"{r['scene']}: "
        f"mIoU={r['mIoU']:.4f}, "
        f"Acc={r['pixel_accuracy']:.4f}"
    )

print("-" * 70)
print(f"Mean scene mIoU       : {aggregate_miou:.4f}")
print(f"Mean scene Pixel Acc. : {aggregate_accuracy:.4f}")
print("=" * 70)

In [ ]:
# ============================================================
# REPRODUCE SAVED SST-UNET RESULTS
# ============================================================

import os
import json
import numpy as np

SAVE_DIR = "/content/SSTUNet_final_reproducible"

print("=" * 70)
print("REPRODUCING SAVED SST-UNET RESULTS")
print("=" * 70)

all_metrics = []

for scene_idx in range(1, len(val_pairs_patch) + 1):

    file_path = os.path.join(
        SAVE_DIR,
        f"scene_{scene_idx}.npz"
    )

    data = np.load(
        file_path,
        allow_pickle=True
    )

    pred = data["prediction"]
    gt = data["ground_truth"]

    miou = compute_miou(
        pred,
        gt,
        NUM_CLASSES
    )

    accuracy = np.mean(pred == gt)

    scene_name = str(data["scene_name"])

    all_metrics.append(miou)

    print(
        f"{scene_name}: "
        f"mIoU={miou:.4f}, "
        f"Pixel Accuracy={accuracy:.4f}"
    )

print("-" * 70)

final_miou = np.mean(all_metrics)

print(f"Reproduced mean scene mIoU = {final_miou:.4f}")
print("=" * 70)

In [ ]:
# ============================================================
# ABLATION STUDY
# 40 epochs + save each trained model for visualization
# ============================================================

RUN_ABLATION = True
ABLATION_EPOCHS = 40

ABLATION_DIR = "/content/ablation_checkpoints"
os.makedirs(ABLATION_DIR, exist_ok=True)


def build_variant(use_band_gate, use_transformer):

    class Variant(nn.Module):

        def __init__(self):
            super().__init__()

            c = BASE_CH

            self.use_bag = use_band_gate
            self.use_trans = use_transformer

            # ----------------------------------------------
            # Spectral Band Attention
            # ----------------------------------------------
            if use_band_gate:
                self.band_gate = BandAttentionGate(N_BANDS)

            # ----------------------------------------------
            # Encoder
            # ----------------------------------------------
            self.enc1 = ConvBlock(
                N_BANDS,
                c
            )

            self.enc2 = ConvBlock(
                c,
                c * 2
            )

            self.enc3 = ConvBlock(
                c * 2,
                c * 4
            )

            self.pool = nn.MaxPool2d(2)

            # ----------------------------------------------
            # Bottleneck
            # ----------------------------------------------
            self.bottleneck_conv = ConvBlock(
                c * 4,
                c * 8
            )

            # ----------------------------------------------
            # Transformer
            # ----------------------------------------------
            if use_transformer:
                self.transformer = (
                    SpectralSpatialTransformerBottleneck(
                        c * 8,
                        N_HEADS,
                        TRANS_DEPTH
                    )
                )

            # ----------------------------------------------
            # Decoder
            # ----------------------------------------------
            self.up3 = nn.ConvTranspose2d(
                c * 8,
                c * 4,
                2,
                stride=2
            )

            self.dec3 = ConvBlock(
                c * 8,
                c * 4
            )

            self.up2 = nn.ConvTranspose2d(
                c * 4,
                c * 2,
                2,
                stride=2
            )

            self.dec2 = ConvBlock(
                c * 4,
                c * 2
            )

            self.up1 = nn.ConvTranspose2d(
                c * 2,
                c,
                2,
                stride=2
            )

            self.dec1 = ConvBlock(
                c * 2,
                c
            )

            # ----------------------------------------------
            # Output heads
            # ----------------------------------------------
            self.seg_head = nn.Conv2d(
                c,
                NUM_CLASSES,
                1
            )

            self.det_head = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Flatten(),
                nn.Linear(c * 8, 64),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(64, NUM_CLASSES),
                nn.Sigmoid()
            )

        def forward(self, x):

            # ------------------------------------------
            # Optional Band Attention
            # ------------------------------------------
            if self.use_bag:
                x, _ = self.band_gate(x)

            # ------------------------------------------
            # Encoder
            # ------------------------------------------
            s1 = self.enc1(x)

            s2 = self.enc2(
                self.pool(s1)
            )

            s3 = self.enc3(
                self.pool(s2)
            )

            # ------------------------------------------
            # Bottleneck
            # ------------------------------------------
            b = self.bottleneck_conv(
                self.pool(s3)
            )

            # ------------------------------------------
            # Optional Transformer
            # ------------------------------------------
            if self.use_trans:
                b = self.transformer(b)

            # ------------------------------------------
            # Detection branch
            # ------------------------------------------
            det = self.det_head(b)

            # ------------------------------------------
            # Decoder
            # ------------------------------------------
            d3 = self.dec3(
                torch.cat(
                    [self.up3(b), s3],
                    dim=1
                )
            )

            d2 = self.dec2(
                torch.cat(
                    [self.up2(d3), s2],
                    dim=1
                )
            )

            d1 = self.dec1(
                torch.cat(
                    [self.up1(d2), s1],
                    dim=1
                )
            )

            return self.seg_head(d1), det, None

    return Variant()


# ============================================================
# Train one ablation variant
# ============================================================

def train_variant(
    name,
    use_band_gate,
    use_transformer,
    epochs
):

    print()
    print("=" * 70)
    print(f"Training: {name}")
    print(
        f"BAG={use_band_gate}, "
        f"Transformer={use_transformer}, "
        f"Epochs={epochs}"
    )
    print("=" * 70)

    model_v = build_variant(
        use_band_gate,
        use_transformer
    ).to(device)

    criterion_v = CombinedLoss(NUM_CLASSES)

    optimizer_v = torch.optim.AdamW(
        model_v.parameters(),
        lr=PATCH_LR,
        weight_decay=1e-4
    )

    scheduler_v = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_v,
        T_max=epochs
    )

    # ----------------------------------------------
    # Training
    # ----------------------------------------------
    for epoch in range(epochs):

        model_v.train()

        running_loss = 0.0

        for cubes, masks, metals in train_patch_loader:

            cubes = cubes.to(device)
            masks = masks.to(device)
            metals = metals.to(device)

            optimizer_v.zero_grad()

            seg_p, det_p, _ = model_v(cubes)

            loss, _, _, _ = criterion_v(
                seg_p,
                masks,
                det_p,
                metals
            )

            loss.backward()

            nn.utils.clip_grad_norm_(
                model_v.parameters(),
                1.0
            )

            optimizer_v.step()

            running_loss += loss.item()

        scheduler_v.step()

        if (
            (epoch + 1) % 10 == 0
            or epoch == 0
            or epoch == epochs - 1
        ):
            print(
                f"Epoch {epoch+1:02d}/{epochs} | "
                f"Loss={running_loss / len(train_patch_loader):.4f}"
            )

    # ----------------------------------------------
    # Full-image evaluation
    # SAME protocol as final model
    # ----------------------------------------------
    model_v.eval()

    v_preds = []
    v_trues = []

    for mat_file, gt_file in val_pairs_patch:

        cube, mask = load_pair(
            os.path.join(DATA_DIR, mat_file),
            os.path.join(DATA_DIR, gt_file)
        )

        _, pred = predict_full_image(
            model_v,
            cube,
            PATCH_SIZE,
            PATCH_STRIDE,
            tta=True
        )

        v_preds.append(
            pred.flatten()
        )

        v_trues.append(
            mask.flatten()
        )

    v_preds = np.concatenate(v_preds)
    v_trues = np.concatenate(v_trues)

    miou = compute_miou(
        v_preds,
        v_trues,
        NUM_CLASSES
    )

    acc = (
        v_preds == v_trues
    ).mean()

    # ----------------------------------------------
    # Save checkpoint
    # ----------------------------------------------
    safe_name = (
        name.lower()
        .replace("+", "plus")
        .replace(" ", "_")
    )

    checkpoint_path = os.path.join(
        ABLATION_DIR,
        f"{safe_name}.pth"
    )

    torch.save(
        model_v.state_dict(),
        checkpoint_path
    )

    print()
    print(f"{name}")
    print(f"mIoU     : {miou:.4f}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Saved    : {checkpoint_path}")

    return model_v, miou, acc, checkpoint_path


# ============================================================
# Run all four variants
# ============================================================

ablation_models = {}
ablation_results = {}

configs = [

    (
        "U-Net only",
        False,
        False
    ),

    (
        "U-Net + Band Attention",
        True,
        False
    ),

    (
        "U-Net + Transformer",
        False,
        True
    ),

    (
        "Full SST-UNet",
        True,
        True
    ),
]


if RUN_ABLATION:

    for name, use_bag, use_trans in configs:

        t0 = time.time()

        model_v, miou, acc, ckpt = train_variant(
            name,
            use_bag,
            use_trans,
            ABLATION_EPOCHS
        )

        ablation_models[name] = model_v

        ablation_results[name] = {
            "mIoU": float(miou),
            "Accuracy": float(acc),
            "seconds": float(time.time() - t0),
            "checkpoint": ckpt
        }

    print()
    print("=" * 70)
    print("FINAL ABLATION RESULTS")
    print("=" * 70)

    for name, result in ablation_results.items():

        print(
            f"{name:28s} "
            f"mIoU={result['mIoU']:.4f}  "
            f"Acc={result['Accuracy']:.4f}"
        )

    with open(
        "/content/ablation_results.json",
        "w"
    ) as f:

        json.dump(
            ablation_results,
            f,
            indent=2
        )

else:

    print(
        "RUN_ABLATION=False — ablation skipped."
    )

In [ ]:
# ============================================================
# CREATE ONE PAPER-READY QUALITATIVE FIGURE
# 3 scenes × 4 columns
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

SAVE_DIR = "/content/SSTUNet_final_reproducible"

CLASS_NAMES = [
    "Background",
    "Copper",
    "Brass",
    "Aluminum",
    "Steel",
    "White Copper"
]

CLASS_COLORS = [
    "#808080",
    "#D95F02",
    "#E6AB02",
    "#1B9E77",
    "#377EB8",
    "#984EA3"
]

cmap = ListedColormap(CLASS_COLORS)


def make_false_rgb(hsi):

    rgb = hsi[:, :, [50, 30, 10]].astype(np.float32)

    for c in range(3):

        lo = np.percentile(rgb[:, :, c], 1)
        hi = np.percentile(rgb[:, :, c], 99)

        rgb[:, :, c] = np.clip(
            (rgb[:, :, c] - lo) /
            (hi - lo + 1e-8),
            0,
            1
        )

    return rgb


# ------------------------------------------------------------
# Load scenes and predictions
# ------------------------------------------------------------

fig, axes = plt.subplots(
    3,
    4,
    figsize=(12, 8)
)

for scene_idx, (mat_file, gt_file) in enumerate(val_pairs_patch):

    # Load original HSI
    cube, _ = load_pair(
        os.path.join(DATA_DIR, mat_file),
        os.path.join(DATA_DIR, gt_file)
    )

    # Load saved prediction
    data = np.load(
        os.path.join(
            SAVE_DIR,
            f"scene_{scene_idx + 1}.npz"
        ),
        allow_pickle=True
    )

    gt = data["ground_truth"]
    pred = data["prediction"]

    miou = compute_miou(
        pred,
        gt,
        NUM_CLASSES
    )

    accuracy = np.mean(pred == gt)

    error = (pred != gt).astype(np.uint8)

    rgb = make_false_rgb(cube)

    # --------------------------------------------------------
    # Column 1: HSI
    # --------------------------------------------------------

    axes[scene_idx, 0].imshow(rgb)
    axes[scene_idx, 0].axis("off")

    # --------------------------------------------------------
    # Column 2: GT
    # --------------------------------------------------------

    axes[scene_idx, 1].imshow(
        gt,
        cmap=cmap,
        vmin=0,
        vmax=5
    )
    axes[scene_idx, 1].axis("off")

    # --------------------------------------------------------
    # Column 3: Prediction
    # --------------------------------------------------------

    axes[scene_idx, 2].imshow(
        pred,
        cmap=cmap,
        vmin=0,
        vmax=5
    )
    axes[scene_idx, 2].axis("off")

    # --------------------------------------------------------
    # Column 4: Error
    # --------------------------------------------------------

    axes[scene_idx, 3].imshow(
        error,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[scene_idx, 3].axis("off")

    # Scene label
    short_name = os.path.splitext(mat_file)[0]

    axes[scene_idx, 0].text(
        -0.02,
        0.5,
        f"Scene {scene_idx + 1}",
        transform=axes[scene_idx, 0].transAxes,
        rotation=90,
        va="center",
        ha="right",
        fontsize=10
    )

# ------------------------------------------------------------
# Column titles
# ------------------------------------------------------------

axes[0, 0].set_title("False-Color HSI", fontsize=11)
axes[0, 1].set_title("Ground Truth", fontsize=11)
axes[0, 2].set_title("SST-UNet Prediction", fontsize=11)
axes[0, 3].set_title("Error Map", fontsize=11)


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

handles = [
    plt.Rectangle(
        (0, 0),
        1,
        1,
        facecolor=CLASS_COLORS[i],
        label=CLASS_NAMES[i]
    )
    for i in range(NUM_CLASSES)
]

fig.legend(
    handles=handles,
    loc="lower center",
    ncol=6,
    frameon=False,
    fontsize=8
)

plt.tight_layout(
    rect=[0.03, 0.08, 1, 1]
)

# ------------------------------------------------------------
# Save paper figure
# ------------------------------------------------------------

figure_path = os.path.join(
    SAVE_DIR,
    "SSTUNet_qualitative_results.png"
)

plt.savefig(
    figure_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(f"\nPaper figure saved at:")
print(figure_path)

In [ ]:
import os
import shutil

source = "/content/SSTUNet_final_reproducible"
destination = "/content/drive/MyDrive/WEEE_checkpoint"

# Create destination folder
os.makedirs(destination, exist_ok=True)

# Copy everything inside the folder
for item in os.listdir(source):
    src = os.path.join(source, item)
    dst = os.path.join(destination, item)

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("Done!")
print("Saved to:", destination)

In [ ]:
import os
import shutil

source = "/content/ablation_checkpoints"
destination = "/content/drive/MyDrive/WEEE_checkpoint"

# Create destination folder
os.makedirs(destination, exist_ok=True)

# Copy everything inside the folder
for item in os.listdir(source):
    src = os.path.join(source, item)
    dst = os.path.join(destination, item)

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("Done!")
print("Saved to:", destination)

In [ ]:
import os
import shutil

source = "/content/qualitative_results"
destination = "/content/drive/MyDrive/WEEE_checkpoint"

# Create destination folder
os.makedirs(destination, exist_ok=True)

# Copy everything inside the folder
for item in os.listdir(source):
    src = os.path.join(source, item)
    dst = os.path.join(destination, item)

    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy2(src, dst)

print("Done!")
print("Saved to:", destination)

In [ ]:
!find /content/drive/MyDrive/WEEE_checkpoint -maxdepth 2 -type f